# 🏍️ AuRoRA-2W-IUR Training Notebook
**Indian Unstructured Roads Multi-Task ADAS Model**

Tasks: Drivable Area Segmentation + Lane Line Segmentation + Object Detection (Vehicles, Potholes, Speed Breakers)

---
**Before running:** Make sure Runtime → Change runtime type → GPU (T4 recommended)

## Step 0: Check GPU & Mount Google Drive

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU')

In [ ]:
# Mount Google Drive (checkpoints will be saved here so they survive session restarts)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/AuRoRA2W_IUR_checkpoints'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_SAVE_DIR}')

## Step 1: Clone the Repository

In [ ]:
# Clone the phase2 branch of the AuRoRA-2W repo
!git clone -b phase2-hybridnets-iur https://github.com/alt-Anurag/AuRoRA-2W.git /content/AuRoRA-2W
%cd /content/AuRoRA-2W/phase2
!pip install -r requirements_phase2.txt -q
print('Repository cloned and dependencies installed!')

## Step 2: Download Datasets

You need 3 datasets. Run the cell for each one you have access to.

| Dataset | Size | Link |
|---|---|---|
| IDD (Indian Driving Dataset) | ~15 GB | https://idd.insaan.iiit.ac.in/dataset/download/ |
| RDD2022 India (Potholes) | ~2 GB | https://www.kaggle.com/datasets/deepsystemsresearch/road-damage-dataset-2022 |
| BDD100K (Lane Lines) | ~6 GB | https://bdd-data.berkeley.edu/ |

**Recommended order:** Start with RDD2022 (smallest, fastest) → BDD100K → IDD

In [ ]:
# ─── RDD2022 via Official Direct Link (sekilab) ────────────────────────────
# Downloading the India subset directly from the original author's S3 bucket.

import os
import shutil

RDD_ROOT = '/content/datasets/RDD2022'
os.makedirs(RDD_ROOT, exist_ok=True)

!wget -q "https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_India.zip" -O /content/datasets/RDD2022/RDD2022_India.zip
!unzip -q /content/datasets/RDD2022/RDD2022_India.zip -d /content/datasets/RDD2022/

# The zip extracts to a folder named 'India'. 
# The expected structure is RDD_ROOT/India/train/images and RDD_ROOT/India/train/labels

# Validation step
india_images = os.path.join(RDD_ROOT, 'India', 'train', 'images')
india_labels = os.path.join(RDD_ROOT, 'India', 'train', 'labels')

if os.path.exists(india_images) and os.path.exists(india_labels):
    num_images = len(os.listdir(india_images))
    if num_images > 0:
        print(f'✅ RDD2022 (India) successfully downloaded and extracted to: {RDD_ROOT}')
        print(f'   Found {num_images} images in train set.')
        !ls {india_images} | head -5
    else:
        raise ValueError(f'❌ RDD2022 download failed: Images directory {india_images} is empty.')
else:
    raise ValueError(f'❌ RDD2022 download failed: Expected directories {india_images} or {india_labels} not found.')


In [ ]:
# ─── BDD100K via direct link ───────────────────────────────────────────────
# You need to be logged in to https://bdd-data.berkeley.edu/
# After logging in, go to Download and copy the direct download links
# OR use the Kaggle mirror (no login needed):

!kaggle datasets download solesensei/solesensei_bdd100k -p /content/datasets/BDD100K --unzip -q

BDD_ROOT = '/content/datasets/BDD100K'
print(f'BDD100K downloaded to: {BDD_ROOT}')
!ls {BDD_ROOT}

In [ ]:
# ─── IDD via wget (paste the download link from your IIIT-H account email) ─
# After registering at https://idd.insaan.iiit.ac.in/dataset/download/
# you will receive an email with download links. Paste them below.

IDD_ROOT = '/content/datasets/IDD'
os.makedirs(IDD_ROOT, exist_ok=True)

# Replace <YOUR_IDD_LINK> with the actual link from your email
# !wget -q "<YOUR_IDD_SEGMENTATION_LINK>" -O /content/datasets/IDD/IDD_Segmentation.tar.gz
# !tar -xzf /content/datasets/IDD/IDD_Segmentation.tar.gz -C {IDD_ROOT}

print('IDD_ROOT set to:', IDD_ROOT)
print('Uncomment the wget lines above and paste your IDD download link.')

In [ ]:
# ─── Set dataset paths (adjust if your download location differs) ──────────
IDD_ROOT = '/content/datasets/IDD'       # or None if not downloaded yet
RDD_ROOT = '/content/datasets/RDD2022'   # or None if not downloaded yet
BDD_ROOT = '/content/datasets/BDD100K'   # or None if not downloaded yet

# Quick check
import os
for name, path in [('IDD', IDD_ROOT), ('RDD2022', RDD_ROOT), ('BDD100K', BDD_ROOT)]:
    if path and os.path.exists(path):
        print(f'[OK] {name}: {path}')
    else:
        print(f'[--] {name}: not found (will be skipped during training)')

## Step 3: Verify Dataset Loading

In [ ]:
import sys
sys.path.insert(0, '/content/AuRoRA-2W/phase2')

from datasets.composite_dataset import build_composite_dataset, collate_fn
from torch.utils.data import DataLoader

dataset = build_composite_dataset(
    idd_root=IDD_ROOT,
    rdd_root=RDD_ROOT,
    bdd_root=BDD_ROOT,
    split='train',
    target_hw=(384, 640),
)

# Quick sanity check: load one sample
sample = dataset[0]
print('Image shape:     ', sample['image'].shape)
print('Drive mask shape:', sample['drive_mask'].shape)
print('Lane mask shape: ', sample['lane_mask'].shape)
print('Boxes shape:     ', sample['boxes'].shape)
print('Roll angle:      ', sample['roll_angle'])
print('\nDataset is working correctly!')

## Step 4: Verify Model Forward Pass

In [ ]:
import torch
from models.aurora2w_iur import get_aurora2w_iur_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_aurora2w_iur_model(num_det_classes=3, num_drive_classes=3).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params/1e6:.2f}M')

# Test forward pass
dummy_img  = torch.randn(2, 3, 384, 640).to(device)
dummy_roll = torch.tensor([0.0, -20.0]).to(device)

with torch.no_grad():
    out = model(dummy_img, dummy_roll)

print('\nModel output shapes:')
for i, d in enumerate(out['det']):
    print(f'  Detection P{i+3}: {d.shape}')
print(f'  Drivable area:  {out["drive_seg"].shape}')
print(f'  Lane lines:     {out["lane_seg"].shape}')
print('\n[OK] Model forward pass working!')

## Step 5: Start Training 🚀

In [ ]:
# Training configuration — adjust as needed
EPOCHS     = 100
BATCH_SIZE = 8      # Reduce to 4 if OOM on T4
LR         = 1e-3
IMG_H      = 384
IMG_W      = 640
WORKERS    = 2

!python train_AuRoRA-2W-IUR.py \
    --idd-root  {IDD_ROOT}  \
    --rdd-root  {RDD_ROOT}  \
    --bdd-root  {BDD_ROOT}  \
    --epochs    {EPOCHS}    \
    --batch-size {BATCH_SIZE} \
    --lr        {LR}        \
    --img-h     {IMG_H}     \
    --img-w     {IMG_W}     \
    --workers   {WORKERS}   \
    --output-dir {DRIVE_SAVE_DIR} \
    --save-every 5

## Step 6: View TensorBoard Logs

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_SAVE_DIR}/tensorboard

## Step 7: Run Inference on a Video

In [ ]:
import glob

# Find latest checkpoint
ckpts = sorted(glob.glob(f'{DRIVE_SAVE_DIR}/aurora2w_iur_epoch_*.pt'))
if ckpts:
    LATEST_CKPT = ckpts[-1]
    print(f'Using checkpoint: {LATEST_CKPT}')
else:
    print('No checkpoints found yet — run training first!')
    LATEST_CKPT = None

# Upload a test video
# from google.colab import files
# uploaded = files.upload()  # Upload your motorcycle video

In [ ]:
VIDEO_PATH = '/content/your_video.mp4'   # Change this
IMU_CSV    = None                         # Optional: path to synced CSV from Phase 1
OUTPUT_VID = '/content/drive/MyDrive/aurora2w_iur_output.mp4'

if LATEST_CKPT:
    !python test_video_iur.py \
        --video      {VIDEO_PATH}  \
        --checkpoint {LATEST_CKPT} \
        --output     {OUTPUT_VID}  \
        --conf-thresh 0.25
    print(f'Output saved to Google Drive: {OUTPUT_VID}')

## 📋 Troubleshooting

| Error | Fix |
|---|---|
| `CUDA out of memory` | Reduce `BATCH_SIZE` to 4 or 2 |
| `No dataset roots found` | Check that dataset paths exist with `!ls {IDD_ROOT}` |
| `ModuleNotFoundError` | Make sure you're in `/content/AuRoRA-2W/phase2` with `%cd` |
| Session disconnected | Training resumes automatically from latest checkpoint in Drive |
| Colab timeout | Use `/schedule` command in Antigravity to set a keep-alive reminder |